In [145]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [ ]:
JNPFile=r"path/filename.csv"

In [147]:
df_JNP=pd.read_csv(JNPFile)

In [148]:
df_JNP['Key']=df_JNP['Article Number']+df_JNP['Product Group']+df_JNP['Attribute']

In [149]:
df_PNS=df_JNP[['Article Number','Product Group']].drop_duplicates(ignore_index=True)

In [ ]:
df_PNS

## AutocareList

In [ ]:
df_AC=pd.read_csv(r"path/filename.csv")

In [152]:
df_AC=df_AC[[ 'PartTerminologyName','PAName', 'UOMLabel']]

In [153]:
df_AC['PAName'][1]

'Width'

In [ ]:
df_AC.dropna(subset=['PAName']).reset_index(drop=True)

In [155]:
df_AC["Attribute_Fullname"] = np.where(df_AC["UOMLabel"].notna(), df_AC['PAName'] + " ("+df_AC['UOMLabel']+")", df_AC["PAName"])

In [156]:
df_AC=df_AC.drop_duplicates(ignore_index=True)

In [157]:
df_AC['Autocare']="Autocare_Attribute"

In [ ]:
df_AC

## Merging

In [159]:
df_Listed=df_PNS.merge(df_AC,how='left',left_on='Product Group',right_on='PartTerminologyName')


In [ ]:
df_Listed=df_Listed[['Article Number','Product Group', 'Attribute_Fullname',"Autocare"]]
df_Listed

In [161]:
df_Listed['Key']=df_Listed['Article Number']+df_Listed['Product Group']+df_Listed['Attribute_Fullname']

In [162]:
df_Final=df_Listed.merge(df_JNP,how="outer")

In [163]:
df_Final['Attributes']=np.where(df_Final["Attribute_Fullname"].notna(), df_Final["Attribute_Fullname"] , df_Final["Attribute"])

In [164]:
df_Final=df_Final[['Article Number', 'Product Group', 'Attributes', 'Autocare', 'Value']]

In [165]:
df_Final['AC_AttributeCount']=np.where(df_Final["Autocare"].notna(), 1 , 0)

In [166]:
df_Final=df_Final.dropna(subset='Autocare')

In [167]:
df_Final['Attribute_Value']=np.where((df_Final["Autocare"].notna())  & (df_Final["Value"].notna()), 1 , 0)

In [ ]:
df_Final

In [169]:
df_1 = df_Final.iloc[:1000000,:]
df_2 = df_Final.iloc[1000000:,:]

In [170]:
df_Final=df_Final.groupby(['Article Number','Product Group'], as_index=False).agg(
    Total_Attr=('AC_AttributeCount','sum'),
    Filled_Attr=('Attribute_Value','sum'))

In [171]:
df_Final['Filled%']=df_Final['Filled_Attr']/df_Final['Total_Attr']

In [ ]:
df_Final

## Exporting

In [ ]:
with pd.ExcelWriter(r'path/filename.xlsx') as writer:  # doctest: +SKIP
    df_1.to_excel(writer,index=False, sheet_name='Raw1')
    df_2.to_excel(writer,index=False, sheet_name='Raw2')
    df_Final.to_excel(writer,index=False, sheet_name='Fillinginfo')